####To study and implement various intensity transformation techniques in spatial domain such as

1. Contrash Stretching
2. Log Transformation
3. Power-law(Gamma) Transformation
4. Thresholding

####and anaylyse their effect

In [1]:
# Install necessary libraries
!pip install ipywidgets

SyntaxError: invalid syntax (3960477375.py, line 2)

In [2]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [3]:
# Load and display original image
import requests
import numpy as np

# Download a sample image directly into memory
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e0/Clouds_over_the_Black_Mountains_from_the_Sugar_Loaf_-_geograph.org.uk_-_1789182.jpg/640px-Clouds_over_the_Black_Mountains_from_the_Sugar_Loaf_-_geograph.org.uk_-_1789182.jpg"
response = requests.get(image_url)
image_bytes = np.asarray(bytearray(response.content), dtype=np.uint8)
img = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

# Check if image loading was successful
if img is None:
    print("Error: Could not load image from URL.")
else:
    img_RGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_Gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Convert to float for further transformations
    img_float = img_Gray.astype(np.float32) / 255.0 # Normalize to 0-1 range for gamma and contrast stretching

    plt.figure(figsize=(6,5))
    plt.imshow(img_Gray, cmap = 'gray')
    plt.title('Original Grayscale Image')
    plt.axis('off')
    plt.show()

    print(img_Gray.shape)


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [4]:
# Negative Transformation
r = np.arange(256)

s = 255 - r

negative = 255 - img_Gray

plt. figure(figsize=(12,5))


plt. subplot (1,3,1)
plt.plot(r, s)
plt. title("Negative Transformation Curve")
plt.xlabel ("Input Intensity (r)")
plt.ylabel ("Output Intensity (s)")
plt.grid()

plt.subplot (1,3,2)
plt.imshow(img_Gray, cmap = 'gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot (1,3,3)
plt.imshow(negative, cmap = 'gray')
plt.title('Negative Image')
plt.axis('off')

plt.show()

NameError: name 'img_Gray' is not defined

In [5]:
# Log Transformation
# Convert to float is already done globally in PCle5hFX4IiT as img_float

# Create input intensity values
r = np.linspace(0, 255, 256)

# Compute scaling constant
c = 255 / np.log(1 + 255)

# Log curve
s = c * np.log(1 + r)

# Apply log transformation to image (using normalized img_float)
# Ensure img_float is scaled back to 0-255 for log transformation if needed, or adjust formula
# For log transformation, it's often applied to 0-255 values, so we might need to denormalize img_float if it's 0-1
# Let's assume the original range 0-255 for log transformation as per common practice.
# If img_float from PCle5hFX4IiT is 0-1, we need to multiply by 255 first.
log_transformed = c * np.log(1 + (img_float * 255))
log_transformed = np.uint8(log_transformed)

# Plot
plt.figure(figsize=(12,5))

plt.subplot(1,3,1)
plt.plot(r, s)
plt.title("Log Transformation Curve")
plt.xlabel("Input Intensity")
plt.ylabel("Output Intensity")
plt.grid()

plt.subplot(1,3,2)
plt.imshow(img_Gray, cmap='gray')
plt.title("Original")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(log_transformed, cmap='gray')
plt.title("Log Transformed")
plt.axis('off')

plt.show()

NameError: name 'img_Gray' is not defined

In [6]:
# Power-law (Gamma) Transformation
def gamma_transform(gamma):

    # Gamma correction (img_float is already 0-1)
    gamma_img = np.power(img_float, gamma)
    gamma_img_uint8 = np.uint8(gamma_img * 255) # Scale back to 0-255 for display

    # Curve (using normalized input r from 0 to 1)
    r_curve = np.linspace(0, 1, 256)
    s_curve = np.power(r_curve, gamma)

    plt.figure(figsize=(15,10))

    #Gamma Curve
    plt.subplot(2,2,1)
    plt.plot(r_curve, s_curve)
    plt.title(f"Gamma Curve (γ={gamma})")
    plt.xlabel("Input Intensity (Normalized)")
    plt.ylabel("Output Intensity (Normalized)")
    plt.grid()

    # Original Image
    plt.subplot(2,2,2)
    plt.imshow(img_Gray, cmap='gray')
    plt.title("Original Image")
    plt.axis('off')

    #Gamma Corrected Image
    plt.subplot(2,2,3)
    plt.imshow(gamma_img_uint8, cmap='gray')
    plt.title("Gamma Corrected Image")
    plt.axis('off')

    # Histogram After Gamma
    plt.subplot(2,2,4)
    plt.hist(gamma_img_uint8.ravel(), bins=256)
    plt.title("Histogram After Gamma")
    plt.xlabel("Pixel Intensity")
    plt.ylabel("Frequency")

    plt.tight_layout()
    plt.show()


gamma_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Gamma'
)

widgets.interact(gamma_transform, gamma=gamma_slider)

interactive(children=(FloatSlider(value=1.0, description='Gamma', max=5.0, min=0.1), Output()), _dom_classes=(…

<function __main__.gamma_transform(gamma)>

In [7]:
# Contrast Stretching
# Find min and max intensity (img_float is already 0-1)
r_min = np.min(img_float)
r_max = np.max(img_float)

# Avoid division by zero
if r_max == r_min:
    contrast_img = img_float.copy()
else:
    contrast_img = (img_float - r_min) / (r_max - r_min) # Result is 0-1

contrast_img = np.uint8(contrast_img * 255) # Scale back to 0-255 for display

# Transformation curve (input r from 0-1, output s from 0-1)
r_curve = np.linspace(0, 1, 256)
s_curve = (r_curve - r_min) / (r_max - r_min)
s_curve[s_curve < 0] = 0 # Clip values outside 0-1 range
s_curve[s_curve > 1] = 1 # Clip values outside 0-1 range

# Plot
plt.figure(figsize=(12,5))

plt.subplot(1,3,1)
plt.plot(r_curve, s_curve * 255) # Plot curve in 0-255 scale for clarity
plt.title("Contrast Stretching Curve")
plt.xlabel("Input Intensity (Normalized)")
plt.ylabel("Output Intensity (Normalized)")
plt.grid()

plt.subplot(1,3,2)
plt.imshow(img_Gray, cmap='gray')
plt.title("Original")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(contrast_img, cmap='gray')
plt.title("Contrast Stretched")
plt.axis('off')

plt.show()

NameError: name 'img_float' is not defined

In [8]:
# Thresholding
def threshold_transform(T):

    # Apply threshold
    thresh_img = np.where(img_Gray > T, 255, 0)
    thresh_img = np.uint8(thresh_img)

    plt.figure(figsize=(12,4))

    # Original Image
    plt.subplot(1,3,1)
    plt.imshow(img_Gray, cmap='gray')
    plt.title("Original Image")
    plt.axis('off')

    # Threshold Curve (Step Function)
    r = np.linspace(0,255,256)
    s = np.where(r > T, 255, 0)

    plt.subplot(1,3,2)
    plt.plot(r, s)
    plt.title("Threshold Curve")
    plt.xlabel("Input Intensity")
    plt.ylabel("Output Intensity")

    # Threshold Output
    plt.subplot(1,3,3)
    plt.imshow(thresh_img, cmap='gray')
    plt.title(f"Threshold = {T}")
    plt.axis('off')

    plt.show()

# Slider
threshold_slider = widgets.IntSlider(
    value=127,
    min=0,
    max=255,
    step=1,
    description='Threshold:'
)

widgets.interact(threshold_transform, T=threshold_slider)

interactive(children=(IntSlider(value=127, description='Threshold:', max=255), Output()), _dom_classes=('widge…

<function __main__.threshold_transform(T)>